# Day 4: Fund Performance Analytics

This notebook contains the complete financial analytics pipeline to calculate, evaluate, and rank 40 mutual fund schemes based on historical NAV daily data (2022-2026) and benchmark indices.

## Objectives:
1. **Compute Daily Returns** and validate their distributions.
2. **Compute CAGR** for 1-year, 3-year, and Max Available Period (~4.4 years).
3. **Compute Sharpe Ratio** (annualized, Rf = 6.5%).
4. **Compute Sortino Ratio** (annualized downside risk adjusted).
5. **Compute Alpha & Beta** (OLS regression against NIFTY 100 benchmark).
6. **Compute Maximum Drawdown** and identify the worst peak-to-trough periods.
7. **Build Fund Scorecard** (composite score 0-100 based on weighted rankings).
8. **Benchmark Comparison Chart** (cumulative returns plot for Top 5 Funds vs Nifty 50 & Nifty 100).


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 12, 'figure.titlesize': 14})

db_path = "../bluestock_mf.db"
conn = sqlite3.connect(db_path)
print("Connected to Database successfully.")

## Step 1: Extract Data from SQLite
We will extract fund master details, historical NAVs, and index values from our SQLite database.

In [ ]:
# Load fund master details
df_funds = pd.read_sql_query("SELECT amfi_code, fund_house, scheme_name, category, plan, expense_ratio_pct FROM dim_fund;", conn)

# Load daily NAV values
df_nav_raw = pd.read_sql_query("SELECT amfi_code, date, nav FROM fact_nav ORDER BY amfi_code, date;", conn)
df_nav_raw['date'] = pd.to_datetime(df_nav_raw['date'])

# Load benchmark indices
df_bench_raw = pd.read_sql_query("SELECT date, index_name, close_value FROM benchmark_indices ORDER BY index_name, date;", conn)
df_bench_raw['date'] = pd.to_datetime(df_bench_raw['date'])

print(f"Loaded {len(df_funds)} funds, {len(df_nav_raw)} NAV rows, and {len(df_bench_raw)} benchmark rows.")

## Step 2: Compute Daily Returns
We calculate daily returns for all funds:
$$daily\_return = \frac{NAV_t}{NAV_{t-1}} - 1$$
We will validate that the returns distributions look reasonable across all funds.

In [ ]:
# Calculate daily returns
df_nav_raw['daily_return'] = df_nav_raw.groupby('amfi_code')['nav'].pct_change()

# Calculate NIFTY100 daily returns
df_n100 = df_bench_raw[df_bench_raw['index_name'] == 'NIFTY100'].copy().sort_values('date')
df_n100['n100_return'] = df_n100['close_value'].pct_change()

# Calculate NIFTY50 daily returns
df_n50 = df_bench_raw[df_bench_raw['index_name'] == 'NIFTY50'].copy().sort_values('date')
df_n50['n50_return'] = df_n50['close_value'].pct_change()

# Group statistics to validate distributions
df_stats = df_nav_raw.groupby('amfi_code')['daily_return'].agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
df_stats = pd.merge(df_stats, df_funds[['amfi_code', 'scheme_name']], on='amfi_code')

print("Summary of daily returns distribution stats across all 40 funds:")
print(df_stats[['mean', 'std', 'min', 'max']].describe())

## Step 3: Compute Financial Performance Metrics
For each fund, we compute:
* **CAGR** for 1-year, 3-year, and Max available period (~4.4 years):
$$CAGR = \left(\frac{NAV_{end}}{NAV_{start}}\right)^{\frac{1}{n}} - 1$$
* **Annualized Sharpe Ratio**: Annual excess return over the risk-free rate ($Rf = 6.5\%$) divided by annualized return volatility:
$$Sharpe = \frac{E(R_p) - R_{f, daily}}{Std(R_p)} \times \sqrt{252}$$
* **Annualized Sortino Ratio**: Downside deviation instead of total standard deviation in the denominator:
$$Sortino = \frac{E(R_p) - R_{f, daily}}{\sqrt{E(\min(R_p, 0)^2)}} \times \sqrt{252}$$
* **Alpha and Beta**: Annualized Alpha and Beta calculated using OLS regression against the daily returns of Nifty 100 benchmark.
* **Maximum Drawdown**: Worst peak-to-trough drawdown and its date range:
$$Drawdown = \frac{NAV}{Running\,Max} - 1$$

In [ ]:
fund_metrics = []

for idx, fund in df_funds.iterrows():
    amfi_code = fund['amfi_code']
    scheme_name = fund['scheme_name']
    
    # Slice NAV for current fund
    fund_nav = df_nav_raw[df_nav_raw['amfi_code'] == amfi_code].copy().sort_values('date')
    daily_rets = fund_nav['daily_return'].dropna()
    latest_date = fund_nav['date'].max()
    latest_nav = fund_nav.loc[fund_nav['date'] == latest_date, 'nav'].values[0]
    
    # Closest date lookup helper
    def get_nav_at_offset(years):
        target_date = latest_date - pd.DateOffset(years=years)
        diffs = (fund_nav['date'] - target_date).abs()
        closest_idx = diffs.idxmin()
        closest_row = fund_nav.loc[closest_idx]
        actual_years = (latest_date - closest_row['date']).days / 365.25
        return closest_row['nav'], actual_years

    # CAGR 1yr & 3yr
    nav_1yr, yrs_1yr = get_nav_at_offset(1)
    cagr_1yr = (latest_nav / nav_1yr) ** (1.0 / yrs_1yr) - 1.0
    
    nav_3yr, yrs_3yr = get_nav_at_offset(3)
    cagr_3yr = (latest_nav / nav_3yr) ** (1.0 / yrs_3yr) - 1.0
    
    # Max Available Period CAGR (~4.4 years)
    first_nav_val = fund_nav.iloc[0]['nav']
    yrs_max = (latest_date - fund_nav.iloc[0]['date']).days / 365.25
    cagr_max = (latest_nav / first_nav_val) ** (1.0 / yrs_max) - 1.0
    
    # Sharpe Ratio
    rf_daily = 0.065 / 252
    excess_rets = daily_rets - rf_daily
    std_ret = daily_rets.std()
    sharpe = (excess_rets.mean() / std_ret) * np.sqrt(252) if std_ret > 0 else np.nan
    
    # Sortino Ratio
    downside_rets = np.minimum(daily_rets, 0)
    downside_std = np.sqrt(np.mean(downside_rets ** 2))
    sortino = (excess_rets.mean() / downside_std) * np.sqrt(252) if downside_std > 0 else np.nan
    
    # Alpha & Beta (OLS against Nifty 100 daily returns)
    df_reg = pd.merge(fund_nav[['date', 'daily_return']], df_n100[['date', 'n100_return']], on='date').dropna()
    slope, intercept, r_val, p_val, std_err = linregress(df_reg['n100_return'], df_reg['daily_return'])
    beta = slope
    alpha = intercept * 252 # Annualized Alpha
    r_squared = r_val ** 2
    
    # Maximum Drawdown
    running_max = fund_nav['nav'].cummax()
    drawdowns = fund_nav['nav'] / running_max - 1.0
    max_dd = drawdowns.min()
    
    trough_idx = drawdowns.idxmin()
    trough_date = fund_nav.loc[trough_idx, 'date']
    peak_idx = fund_nav.loc[:trough_idx, 'nav'].idxmax()
    peak_date = fund_nav.loc[peak_idx, 'date']
    
    fund_metrics.append({
        'amfi_code': amfi_code,
        'scheme_name': scheme_name,
        'fund_house': fund['fund_house'],
        'category': fund['category'],
        'plan': fund['plan'],
        'expense_ratio_pct': fund['expense_ratio_pct'],
        'cagr_1yr_pct': cagr_1yr * 100,
        'cagr_3yr_pct': cagr_3yr * 100,
        'cagr_max_pct': cagr_max * 100,
        'sharpe_ratio': sharpe,
        'sortino_ratio': sortino,
        'beta': beta,
        'alpha_pct': alpha * 100,
        'r_squared': r_squared,
        'max_drawdown_pct': max_dd * 100,
        'worst_dd_start': peak_date.strftime('%Y-%m-%d'),
        'worst_dd_end': trough_date.strftime('%Y-%m-%d')
    })

df_metrics = pd.DataFrame(fund_metrics)
print("Computed financial performance metrics for all 40 funds.")
df_metrics[['scheme_name', 'cagr_3yr_pct', 'sharpe_ratio', 'alpha_pct', 'max_drawdown_pct']].head()

## Step 4: Build Fund Scorecard (0-100)
We will rank all 40 funds on percentile bases (0-100) and construct a composite score:
$$Composite\,Score = 30\% \times 3yr\,Return\,Rank + 25\% \times Sharpe\,Rank + 20\% \times Alpha\,Rank + 15\% \times Expense\,Ratio\,Rank\,(inverse) + 10\% \times Max\,DD\,Rank\,(inverse)$$

In [ ]:
# Rank funds
df_metrics['rank_3yr_cagr'] = df_metrics['cagr_3yr_pct'].rank(pct=True) * 100
df_metrics['rank_sharpe'] = df_metrics['sharpe_ratio'].rank(pct=True) * 100
df_metrics['rank_alpha'] = df_metrics['alpha_pct'].rank(pct=True) * 100
df_metrics['rank_expense'] = df_metrics['expense_ratio_pct'].rank(ascending=False, pct=True) * 100
df_metrics['rank_max_dd'] = df_metrics['max_drawdown_pct'].rank(pct=True) * 100

# Compute composite score
df_metrics['fund_score'] = (
    0.30 * df_metrics['rank_3yr_cagr'] +
    0.25 * df_metrics['rank_sharpe'] +
    0.20 * df_metrics['rank_alpha'] +
    0.15 * df_metrics['rank_expense'] +
    0.10 * df_metrics['rank_max_dd']
)

# Sort by composite score
df_scorecard = df_metrics.sort_values(by='fund_score', ascending=False).reset_index(drop=True)
def get_tier(score):
    if score >= 80: return 'Tier 1 (Excellent)'
    elif score >= 60: return 'Tier 2 (Good)'
    elif score >= 40: return 'Tier 3 (Average)'
    else: return 'Tier 4 (Underperforming)'
df_scorecard['performance_tier'] = df_scorecard['fund_score'].apply(get_tier)

print("Top 10 Funds by Scorecard Score:")
df_scorecard[['scheme_name', 'cagr_3yr_pct', 'sharpe_ratio', 'alpha_pct', 'expense_ratio_pct', 'fund_score', 'performance_tier']].head(10)

## Step 5: Benchmark Comparison and Tracking Error
We select the top 5 funds from our scorecard and plot their 3-year cumulative returns normalized to base 100 on `2023-05-29` against the `NIFTY50` and `NIFTY100` indices. We also calculate the annualized Tracking Error:
$$Tracking\,Error = Std(R_{fund} - R_{benchmark}) \times \sqrt{252}$$

In [ ]:
top_5 = df_scorecard.head(5)
top_5_amfi = top_5['amfi_code'].tolist()
top_5_names = top_5['scheme_name'].tolist()

# Date window
end_date = pd.to_datetime('2026-05-29')
start_date = end_date - pd.DateOffset(years=3)

# Slice data
df_nav_3yr = df_nav_raw[(df_nav_raw['date'] >= start_date) & (df_nav_raw['date'] <= end_date)].copy()
df_n100_3yr = df_n100[(df_n100['date'] >= start_date) & (df_n100['date'] <= end_date)].copy().sort_values('date')
df_n50_3yr = df_n50[(df_n50['date'] >= start_date) & (df_n50['date'] <= end_date)].copy().sort_values('date')

plt.figure(figsize=(12, 7))

tracking_errors = []

for code, name in zip(top_5_amfi, top_5_names):
    df_f = df_nav_3yr[df_nav_3yr['amfi_code'] == code].copy().sort_values('date')
    first_nav = df_f.iloc[0]['nav']
    df_f['normalized_nav'] = (df_f['nav'] / first_nav) * 100
    
    # Plot
    plt.plot(df_f['date'], df_f['normalized_nav'], label=name.split(" - ")[0], linewidth=1.5)
    
    # Align
    df_align = pd.merge(df_f[['date', 'daily_return']], df_n100_3yr[['date', 'n100_return']], on='date').dropna()
    df_align = pd.merge(df_align, df_n50_3yr[['date', 'n50_return']], on='date').dropna()
    
    te_n100 = np.std(df_align['daily_return'] - df_align['n100_return']) * np.sqrt(252) * 100
    te_n50 = np.std(df_align['daily_return'] - df_align['n50_return']) * np.sqrt(252) * 100
    
    tracking_errors.append({
        'scheme_name': name,
        'tracking_error_vs_nifty100_pct': te_n100,
        'tracking_error_vs_nifty50_pct': te_n50
    })

# Normalize benchmarks
n100_start = df_n100_3yr.iloc[0]['close_value']
df_n100_3yr['normalized_close'] = (df_n100_3yr['close_value'] / n100_start) * 100

n50_start = df_n50_3yr.iloc[0]['close_value']
df_n50_3yr['normalized_close'] = (df_n50_3yr['close_value'] / n50_start) * 100

plt.plot(df_n100_3yr['date'], df_n100_3yr['normalized_close'], label='NIFTY 100 (Benchmark)', color='black', linewidth=2.5, linestyle='--')
plt.plot(df_n50_3yr['date'], df_n50_3yr['normalized_close'], label='NIFTY 50 (Benchmark)', color='red', linewidth=2.5, linestyle=':')

plt.title("3-Year Cumulative Return Comparison: Top 5 Funds vs Benchmarks (Base 100)")
plt.xlabel("Date")
plt.ylabel("Normalized Value (Starting at 100)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Show tracking errors
df_te = pd.DataFrame(tracking_errors)
print("Tracking Errors for Top 5 Funds:")
df_te

## Step 6: Close Database Connection